# Digit Classification with MLP and CNN in PyTorch

This project trains and compares two neural network classifiers — a **multi-layer perceptron (MLP)** and a **convolutional neural network (CNN)** — on the scikit-learn `digits` dataset (8x8 grayscale digit images, 10 classes).

The training pipeline is written from first principles in PyTorch and includes:
- a custom `Dataset` class,
- a unified `ClassifierNeuralNet` wrapper that produces `LogSoftmax` outputs and exposes a `forward()` returning the NLL loss,
- a training loop with **early stopping** based on validation loss,
- a clean evaluation routine reporting NLL and classification error on the held-out test set.

## 1. Setup

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits
from torch.utils.data import DataLoader, Dataset

EPS = 1.0e-7
torch.manual_seed(0)
np.random.seed(0)

results_dir = './results/'
os.makedirs(results_dir, exist_ok=True)

## 2. Dataset

In [ ]:
class Digits(Dataset):
    """Wrapper around the scikit-learn 8x8 digits dataset.

    Splits:
        train: indices  [0:1000]
        val:   indices  [1000:1350]
        test:  indices  [1350:]
    """

    def __init__(self, mode='train', transforms=None):
        digits = load_digits()
        if mode == 'train':
            self.data = digits.data[:1000].astype(np.float32)
            self.targets = digits.target[:1000]
        elif mode == 'val':
            self.data = digits.data[1000:1350].astype(np.float32)
            self.targets = digits.target[1000:1350]
        else:
            self.data = digits.data[1350:].astype(np.float32)
            self.targets = digits.target[1350:]
        self.transforms = transforms

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample_x = self.data[idx]
        sample_y = self.targets[idx]
        if self.transforms:
            sample_x = self.transforms(sample_x)
        return sample_x, sample_y

In [ ]:
# Visualize a few examples
digits = load_digits()
x = digits.data[:16].astype(np.float32)

fig, axs = plt.subplots(4, 4, figsize=(4, 4))
fig.suptitle('Sample digits (8x8)', y=1.02)
fig.tight_layout()
for i in range(4):
    for j in range(4):
        img = np.reshape(x[4 * i + j], (8, 8))
        axs[i, j].imshow(img, cmap='gray')
        axs[i, j].axis('off')
plt.show()

## 3. Helper layers and the classifier wrapper

The dataset stores each image as a flat 64-vector. To use convolutional layers we need to reshape it to a `(1, 8, 8)` tensor; to feed a fully-connected head we need to flatten back to 64. The two helper modules below do exactly that.

In [ ]:
class Reshape(nn.Module):
    """Reshape a flat input to a tensor of the given shape (excluding batch)."""

    def __init__(self, size):
        super().__init__()
        self.size = size

    def forward(self, x):
        assert x.shape[1] == int(np.prod(self.size))
        return x.view(x.shape[0], *self.size)


class Flatten(nn.Module):
    """Flatten everything past the batch dimension."""

    def forward(self, x):
        return x.view(x.shape[0], -1)

### Classifier wrapper

The classifier defines:
- `classify(x)` which returns the predicted class index (used at evaluation time),
- `forward(x, y)` which returns the **negative log-likelihood** loss, mathematically:

$$
\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N} \log p(y_i \mid \mathbf{x}_i)
$$

Because the network outputs `LogSoftmax`, we use `F.nll_loss` (combined with `log_softmax` defensively in case the wrapped network is changed later).

In [ ]:
class ClassifierNeuralNet(nn.Module):
    """Generic classifier that wraps any LogSoftmax-producing classnet."""

    def __init__(self, classnet):
        super().__init__()
        self.classnet = classnet
        self.nll = nn.NLLLoss(reduction='none')

    def classify(self, x):
        y_pred = self.classnet(x)
        return torch.argmax(y_pred, dim=1)

    def forward(self, x, y, reduction='avg'):
        y_pred = self.classnet(x)
        # log_softmax of an already-LogSoftmax output is a no-op up to numerics
        loss = F.nll_loss(F.log_softmax(y_pred, dim=1), y, reduction='none')
        return loss.sum() if reduction == 'sum' else loss.mean()

**A note on `LogSoftmax` vs `Softmax`.** Working in log-space is numerically more stable and pairs naturally with NLL loss. For *predicting* the class label `argmax` is invariant to the monotonic log transformation, so it makes no difference whether we apply softmax before taking the argmax.

## 4. Training and evaluation

In [ ]:
def evaluation(test_loader, name=None, model_best=None, epoch=None):
    """Evaluate the best saved model (or a passed-in model) on a dataloader."""
    if model_best is None:
        model_best = torch.load(name + '.model', weights_only=False)

    model_best.eval()
    loss_test = 0.0
    loss_error = 0.0
    N = 0
    for batch, targets in test_loader:
        loss_test += model_best.forward(batch, targets, reduction='sum').item()
        y_pred = model_best.classify(batch)
        loss_error += (y_pred != targets).sum().item()
        N += batch.shape[0]

    loss_test /= N
    loss_error /= N

    if epoch is None:
        print(f'-> FINAL PERFORMANCE: nll={loss_test:.4f}, ce={loss_error:.4f}')
    elif epoch % 10 == 0:
        print(f'Epoch: {epoch}, val nll={loss_test:.4f}, val ce={loss_error:.4f}')

    return loss_test, loss_error


def plot_curve(name, signal, file_name='curve.pdf', xlabel='epochs',
               ylabel='nll', color='b-', test_eval=None):
    plt.figure()
    plt.plot(np.arange(len(signal)), signal, color, linewidth=3, label=f'{ylabel} val')
    if test_eval is not None:
        plt.hlines(test_eval, xmin=0, xmax=len(signal), linestyles='dashed',
                   label=f'{ylabel} test')
        plt.text(len(signal), test_eval, f'{test_eval:.3f}')
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend()
    plt.savefig(name + file_name, bbox_inches='tight')
    plt.show()

In [ ]:
def training(name, max_patience, num_epochs, model, optimizer,
             training_loader, val_loader):
    """Train with early stopping based on validation NLL."""
    nll_val = []
    error_val = []
    best_nll = float('inf')
    patience = 0

    for e in range(num_epochs):
        model.train()
        for batch, targets in training_loader:
            loss = model.forward(batch, targets)
            optimizer.zero_grad()
            loss.backward(retain_graph=True)
            optimizer.step()

        loss_e, error_e = evaluation(val_loader, model_best=model, epoch=e)
        nll_val.append(loss_e)
        error_val.append(error_e)

        # Early stopping: save the best model, stop if no progress
        if e == 0 or loss_e < best_nll:
            torch.save(model, name + '.model')
            best_nll = loss_e
            patience = 0
        else:
            patience += 1

        if patience > max_patience:
            print(f'Early stopping at epoch {e}')
            break

    return np.asarray(nll_val), np.asarray(error_val)

## 5. Data loaders

In [ ]:
train_data = Digits(mode='train')
val_data = Digits(mode='val')
test_data = Digits(mode='test')

training_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

print(f'Train / Val / Test sizes: {len(train_data)} / {len(val_data)} / {len(test_data)}')

example_x, example_y = train_data[1]
print(f'Feature shape: {example_x.shape}, label: {example_y}')

train_features, train_labels = next(iter(training_loader))
print(f'Feature batch shape: {train_features.size()}, labels batch shape: {train_labels.size()}')

reshape = Reshape(size=(1, 8, 8))
flatten = Flatten()
print(f'After Reshape((1, 8, 8)): {reshape(train_features).size()}')
print(f'After Flatten():           {flatten(reshape(train_features)).size()}')

## 6. Hyperparameters

In [ ]:
# Data
D = 64           # input dimension (8 * 8)

# Model
M = 256          # MLP hidden width
K = 10           # number of classes
num_kernels = 32 # CNN feature maps in the second conv block

# Training
lr = 1e-3
wd = 1e-5
num_epochs = 1000
max_patience = 20

## 7. Architectures and training

We define and train two networks:

- **MLP**: `Flatten -> Linear(64, 256) -> ReLU -> Dropout -> Linear(256, 10) -> LogSoftmax`. Standard fully-connected classifier; treats the image as a 64-vector and ignores spatial structure.
- **CNN**: `Reshape(1, 8, 8) -> Conv -> ReLU -> MaxPool -> Conv -> ReLU -> MaxPool -> BatchNorm -> Flatten -> Linear -> ReLU -> Dropout -> Linear -> LogSoftmax`. Exploits the 2D structure of the image with two convolutional blocks before the fully-connected head.

In [ ]:
names = ['classifier_mlp', 'classifier_cnn']
val_curves = {}

for name in names:
    print(f'\n-> START {name}')

    if name == 'classifier_mlp':
        run_name = f'{name}_M_{M}'
        classnet = nn.Sequential(
            Flatten(),
            nn.Linear(D, M),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(M, K),
            nn.LogSoftmax(dim=1),
        )
    else:  # classifier_cnn
        run_name = f'{name}_M_{M}_kernels_{num_kernels}'
        classnet = nn.Sequential(
            Reshape(size=(1, 8, 8)),
            nn.Conv2d(in_channels=1, out_channels=16,
                      kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(in_channels=16, out_channels=num_kernels,
                      kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.BatchNorm2d(num_kernels),
            Flatten(),
            nn.Linear(num_kernels * 2 * 2, M),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(M, K),
            nn.LogSoftmax(dim=1),
        )

    model = ClassifierNeuralNet(classnet)
    optimizer = torch.optim.Adamax(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr, weight_decay=wd,
    )

    nll_val, error_val = training(
        name=os.path.join(results_dir, run_name),
        max_patience=max_patience,
        num_epochs=num_epochs,
        model=model,
        optimizer=optimizer,
        training_loader=training_loader,
        val_loader=val_loader,
    )

    test_loss, test_error = evaluation(
        name=os.path.join(results_dir, run_name),
        test_loader=test_loader,
    )

    with open(os.path.join(results_dir, run_name + '_test_loss.txt'), 'w') as f:
        f.write(f'NLL: {test_loss}\nCE: {test_error}\n')

    val_curves[run_name] = (nll_val, error_val, test_loss, test_error)

    plot_curve(os.path.join(results_dir, run_name), nll_val,
               file_name='_nll_val_curve.pdf', ylabel='nll', test_eval=test_loss)
    plot_curve(os.path.join(results_dir, run_name), error_val,
               file_name='_ca_val_curve.pdf', ylabel='ce', color='r-',
               test_eval=test_error)

## 8. Analysis

**Convergence of MLP vs CNN.** The CNN typically converges to a lower validation loss and lower classification error than the MLP, and tends to generalize slightly better on the held-out test set. The MLP overfits more quickly: validation loss starts ticking back up while training loss is still falling.

**Why does the CNN do better on images?**
- **Translation equivariance:** convolutional filters apply the same weights across the image, so a stroke detected in one corner is detected the same way in another corner. An MLP would have to learn this from scratch for each spatial location.
- **Local connectivity + parameter sharing:** dramatically fewer parameters than a fully-connected layer of the same expressive power, which acts as a built-in regularizer.
- **Hierarchical features:** stacked conv-pool blocks build up increasingly abstract features (edges → strokes → digit parts), aligned with how digit classes actually differ.

These advantages are smaller on 8x8 images than on natural-resolution images, but they are still measurable here.